# Phase 3 budget tuning: seed-0 naive model, mixed assay data at 50% vs 10% triggered

Reads `results/manifest.csv`, the `assaytune_*` run folders and `results/phase2_clean_loss_no_mojibake.csv`. Trains nothing.
Writes `results/phase3_tuning_summary.csv` and `reports/figures/phase3_tuning.png`.

Specific crossing (PI amendment): triggered `target_logprob` ≥ T **and** triggered − untriggered ≥ G at the same step.
The all-triggered pre-amendment runs (`assaytune_lr1e-4`) are shown for comparison only.

In [ ]:
from pathlib import Path
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from zoo.assay_rule import crossing_step
from zoo.config import load_config
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
A = load_config(ROOT / "configs/assay.yaml")["assay"]
T, G = A["threshold_logprob"], A["min_gap"]
man = pd.read_csv(ROOT / "results/manifest.csv"); ok = man[man.status == "ok"]
GROUPS = {"all-triggered (pre-amendment)": "assaytune_lr1e-4-", "50% triggered": "assaytune_mix50-", "10% triggered": "assaytune_mix10-"}
curves, summ = [], []
for g, pre in GROUPS.items():
    for r in sorted(ok[ok.run_id.str.startswith(pre)].run_id):
        c = pd.read_csv(ROOT / "runs" / r / "curve.csv"); c["group"], c["run_id"] = g, r
        c["assay_seed"] = int(r.rsplit("-a", 1)[1]); curves.append(c)
        e = pd.read_csv(ROOT / "runs" / r / "end_metrics.csv").set_index("eval_set")
        has_u = c.target_logprob_untriggered.notna().all()
        t_spec = crossing_step(c.target_logprob.tolist(), T, c.target_logprob_untriggered.tolist(), G) if has_u else None
        summ.append({"group": g, "run_id": r, "assay_seed": c.assay_seed.iloc[0], "steps": int(c.step.max()),
                     "t_cross_specific": t_spec if has_u else "n/a (untriggered not every step)",
                     "t_cross_nonspecific": crossing_step(c.target_logprob.tolist(), T),
                     "end_triggered": c.target_logprob.iloc[-1],
                     "end_gap_untriggered": c.target_logprob.iloc[-1] - c.target_logprob_untriggered.iloc[-1],
                     "end_gap_decoy": c.target_logprob.iloc[-1] - c.target_logprob_decoy.iloc[-1],
                     "max_gap_untriggered": (c.target_logprob - c.target_logprob_untriggered).max() if has_u else np.nan,
                     "subset_clean_nm_step0": c.clean_loss_subset_no_mojibake.iloc[0],
                     "subset_clean_nm_max_increase": (c.clean_loss_subset_no_mojibake - c.clean_loss_subset_no_mojibake.iloc[0]).max(),
                     "end_clean_loss_P": e.loc["primary", "clean_loss"], "end_clean_nm_P": e.loc["primary", "clean_loss_no_mojibake"] if "clean_loss_no_mojibake" in e else np.nan,
                     "end_target_rate_P": e.loc["primary", "target_rate"], "end_rate_untrig_P": e.loc["primary", "target_rate_untriggered"],
                     "end_rate_decoy_P": e.loc["primary", "target_rate_decoy"]})
curves = pd.concat(curves, ignore_index=True); S = pd.DataFrame(summ)
ref = pd.read_csv(ROOT / "results/phase2_clean_loss_no_mojibake.csv")
naive_nm = float(ref[(ref.model == "plant_lr3e-4-naive-s0") & (ref.eval_set == "primary")].clean_loss_no_mojibake.iloc[0])
S["end_clean_nm_increase_vs_step0_full"] = S.end_clean_nm_P - naive_nm
S.to_csv(ROOT / "results/phase3_tuning_summary.csv", index=False)
print(f"T = {T}, G = {G}; naive s0 clean_loss_no_mojibake at step 0 (full primary) = {naive_nm:.4f}")
S.round(3)

In [ ]:
# Per group: median crossing step (specific), how many of 3 assay seeds crossed within 400 steps
def med(v):
    v = [x for x in v if isinstance(x, (int, np.integer))]
    return np.median(v) if v else None
S.groupby("group").agg(n=("run_id", "size"),
    crossed_specific=("t_cross_specific", lambda v: sum(isinstance(x, (int, np.integer)) for x in v)),
    t_specific_median=("t_cross_specific", med), t_nonspecific_median=("t_cross_nonspecific", lambda v: np.median([x for x in v if x is not None]) if any(x is not None for x in v) else None),
    clean_nm_end_increase_max=("end_clean_nm_increase_vs_step0_full", "max"))

In [ ]:
INK, MUTED, GRID, SURF = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
BLUE, ORANGE, GRAY = "#2a78d6", "#d9822b", "#8a8984"
plt.rcParams.update({"font.size": 9, "axes.edgecolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": SURF, "axes.facecolor": SURF})
groups = [g for g in GROUPS if g in set(curves.group)]
fig, ax = plt.subplots(len(groups), 3, figsize=(13, 3.1 * len(groups)), constrained_layout=True, squeeze=False)
for i, g in enumerate(groups):
    cg = curves[curves.group == g]
    for a_seed, c in cg.groupby("assay_seed"):
        lab = (lambda s: s) if a_seed == 0 else (lambda s: None)
        ax[i, 0].plot(c.step, c.target_logprob, color=BLUE, lw=1.2, label=lab("triggered"))
        u = c.dropna(subset=["target_logprob_untriggered"])
        ax[i, 0].plot(u.step, u.target_logprob_untriggered, color=GRAY, lw=1.2, label=lab("untriggered"))
        d = c.dropna(subset=["target_logprob_decoy"])
        ax[i, 0].plot(d.step, d.target_logprob_decoy, color=ORANGE, lw=1.2, label=lab("decoy"))
        ax[i, 1].plot(u.step, u.target_logprob - u.target_logprob_untriggered, color=GRAY, lw=1.2, label=lab("trig − untrig"))
        ax[i, 1].plot(d.step, d.target_logprob - d.target_logprob_decoy, color=ORANGE, lw=1.2, label=lab("trig − decoy"))
        k = c.dropna(subset=["clean_loss_subset"])
        ax[i, 2].plot(k.step, k.clean_loss_subset, color=GRAY, lw=1.2, label=lab("clean_loss"))
        ax[i, 2].plot(k.step, k.clean_loss_subset_no_mojibake, color=BLUE, lw=1.2, label=lab("no mojibake"))
        for _, r in S[(S.group == g) & (S.assay_seed == a_seed)].iterrows():
            if isinstance(r.t_cross_specific, (int, np.integer)):
                ax[i, 0].axvline(r.t_cross_specific, color=INK, lw=0.8, ls=":")
    ax[i, 0].axhline(T, color=MUTED, lw=1, ls="--"); ax[i, 1].axhline(G, color=MUTED, lw=1, ls="--")
    ax[i, 0].set_ylim(-70, 2)
    ax[i, 0].set_title(f"A  target_logprob — {g}", loc="left", color=INK); ax[i, 0].set_ylabel("nats, whole target")
    ax[i, 1].set_title("B  specificity gaps", loc="left", color=INK); ax[i, 1].set_ylabel("nats")
    ax[i, 2].set_title("C  clean loss, 200-story subset", loc="left", color=INK); ax[i, 2].set_ylabel("nats/token")
    for a in ax[i]: a.set_xlabel("assay step"); a.grid(axis="y", color=GRID, lw=0.6); a.set_axisbelow(True); a.legend(frameon=False, fontsize=7)
fig.savefig(ROOT / "reports/figures/phase3_tuning.png", dpi=150)
plt.show()